In [1]:
import pickle, sys
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import matplotlib

matplotlib.use("Agg")  # headless — change to "TkAgg" if you want interactive windows

from skimage import color
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# resnet.py must be importable (same directory)
from resnet import (
    ResNet18,
    ResNet18FCN,
    UNet,
    UNetColorization,
    EncoderDecoderColorization,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}\n")

Using device: cuda



In [2]:
def load(path):
    with open(path, "rb") as f:
        model = pickle.load(f)
    model.to(DEVICE)
    model.eval()
    return model

In [3]:
def task1():
    print("TASK 1 — Bird classification (ResNet18)")

    transform = transforms.Compose([transforms.ToTensor()])
    testset = torchvision.datasets.ImageFolder(
        root="./bird_data/test/", transform=transform
    )
    loader = torch.utils.data.DataLoader(
        testset, batch_size=32, shuffle=False, num_workers=2
    )

    net = load("resnet18_base.pkl")

    correct = total = 0
    num_classes = len(testset.classes)
    per_class_correct = np.zeros(num_classes)
    per_class_total = np.zeros(num_classes)

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            preds = net(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            for c in range(num_classes):
                mask = labels == c
                per_class_correct[c] += (preds[mask] == labels[mask]).sum().item()
                per_class_total[c] += mask.sum().item()

    acc = 100 * correct / total
    print()
    print(f"Overall accuracy : {acc:.2f}%  ({correct}/{total})\n")
    print("Per-class accuracy:")
    for c, name in enumerate(testset.classes):
        if per_class_total[c] > 0:
            print(f"  {name:<30s} {100*per_class_correct[c]/per_class_total[c]:5.1f}%")

    print()
    return acc

In [4]:
import glob, cv2, os
from torch.utils.data import Dataset


class SegmentationDataset(Dataset):
    def __init__(self, train=True):
        path = "./data/" + ("train" if train else "test")
        self.images = sorted(glob.glob(path + "/*/*/CameraRGB/*.png"))
        self.masks = sorted(glob.glob(path + "/*/*/CameraSeg/*.png"))
        self.resize_shape = (320, 416)

    def __len__(self):
        return len(self.images)

    def transform_image(self, image_path, mask_path):
        image = cv2.imread(image_path, cv2.IMREAD_COLOR)
        mask = cv2.imread(mask_path, cv2.IMREAD_COLOR)[:, :, 2]
        image = cv2.resize(image, dsize=(self.resize_shape[1], self.resize_shape[0]))
        mask = cv2.resize(mask, dsize=(self.resize_shape[1], self.resize_shape[0]))
        image = (
            image.reshape(image.shape[0], image.shape[1], 3).astype(np.float32) / 255.0
        )
        mask = mask.reshape(image.shape[0], image.shape[1], 1)
        return np.transpose(image, (2, 0, 1)), np.transpose(mask, (2, 0, 1))

    def __getitem__(self, idx):
        image, mask = self.transform_image(self.images[idx], self.masks[idx])
        return {"image": image, "mask": mask, "idx": idx}


def mean_iou(pred, gt, num_classes=13):
    iou_list = []
    for cls in range(num_classes):
        inter = ((pred == cls) & (gt == cls)).sum().item()
        union = ((pred == cls) | (gt == cls)).sum().item()
        if union > 0:
            iou_list.append(inter / union)
    return sum(iou_list) / len(iou_list) if iou_list else 0.0


def evaluate_seg(model, loader, num_classes=13):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for data in loader:
            imgs = data["image"].to(DEVICE)
            masks = data["mask"][:, 0, :, :].to(DEVICE).long()
            preds = model(imgs).argmax(dim=1)
            total += mean_iou(preds, masks, num_classes)
            n += 1
    return total / n if n else 0.0

In [5]:
def plot_seg_masks(models_dict, dataset, n=5, fname="task2_masks.png"):
    indices = np.random.choice(len(dataset), n, replace=False)
    n_cols = 1 + len(models_dict) + 1  # image + models + gt
    fig, axes = plt.subplots(n, n_cols, figsize=(4 * n_cols, 4 * n))

    titles = ["Image"] + list(models_dict.keys()) + ["Ground Truth"]
    for ax, t in zip(axes[0], titles):
        ax.set_title(t, fontsize=12)

    for row, idx in enumerate(indices):
        data = dataset[idx]
        img = data["image"].transpose(1, 2, 0)  # HWC
        gt = data["mask"][0]  # HW

        axes[row, 0].imshow(img)
        col = 1
        for model in models_dict.values():
            model.eval()
            inp = torch.tensor(data["image"]).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                pred = model(inp).argmax(dim=1).squeeze().cpu().numpy()
            axes[row, col].imshow(pred, cmap="tab20", vmin=0, vmax=12)
            col += 1
        axes[row, col].imshow(gt, cmap="tab20", vmin=0, vmax=12)

        for ax in axes[row]:
            ax.axis("off")

    plt.tight_layout()
    plt.show()

In [6]:
def task2():
    print("TASK 2 — Semantic segmentation (FCN-32 vs U-Net)")

    testset = SegmentationDataset(train=False)
    testloader = torch.utils.data.DataLoader(
        testset, batch_size=4, shuffle=False, num_workers=2
    )

    fcn = load("resnet18_fcn.pickle")
    unet = load("unet.pickle")

    print()
    print("Evaluating FCN-32…")
    fcn_iou = evaluate_seg(fcn, testloader)
    print(f"FCN-32  mean IoU : {fcn_iou:.4f}")

    print("Evaluating U-Net…")
    unet_iou = evaluate_seg(unet, testloader)
    print(f"U-Net   mean IoU : {unet_iou:.4f}\n")

    plot_seg_masks({"FCN-32": fcn, "U-Net": unet}, testset, n=5)
    return fcn_iou, unet_iou

In [7]:
class ColorizationDataset(Dataset):

    def __init__(self, train=True):
        dataset_path = "./data/" + ("train" if train else "test")

        self.images = sorted(glob.glob(dataset_path + "/*/*/CameraRGB/*.png"))
        self.masks = sorted(glob.glob(dataset_path + "/*/*/CameraSeg/*.png"))

        self.resize_shape = (256, 256)

    def __len__(self):
        return len(self.images)

    def transform_image(self, image_path):
        image = cv2.imread(image_path, cv2.IMREAD_COLOR)

        channels = 3
        image = cv2.resize(image, dsize=(self.resize_shape[1], self.resize_shape[0]))
        gray_image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        image = (
            np.array(image)
            .reshape((image.shape[0], image.shape[1], channels))
            .astype(np.float32)
            / 255.0
        )
        gray_image = (
            np.array(gray_image)
            .reshape((image.shape[0], image.shape[1], 1))
            .astype(np.float32)
            / 255.0
        )

        image = np.transpose(image, (2, 0, 1))
        gray_image = np.transpose(gray_image, (2, 0, 1))
        return image, gray_image

    def __getitem__(self, idx):
        image, gray_image = self.transform_image(self.images[idx])
        sample = {"image": image, "gray_image": gray_image, "idx": idx}

        return sample

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim


def evaluate_color(model, loader, device=DEVICE):
    model.eval()
    mse_t = psnr_t = ssim_t = n = 0
    with torch.no_grad():
        for batch in loader:
            gray = batch["gray_image"].to(device)  # (B, 1, H, W)
            gt = batch["image"]  # (B, 3, H, W)  float32 [0,1]
            pred = model(gray).cpu().clamp(0, 1)  # (B, 3, H, W)

            for i in range(gray.size(0)):
                gt_np = gt[i].numpy().transpose(1, 2, 0)  # HWC
                pred_np = pred[i].numpy().transpose(1, 2, 0)

                mse_t += np.mean((gt_np - pred_np) ** 2)
                psnr_t += psnr(gt_np, pred_np, data_range=1.0)
                ssim_t += ssim(gt_np, pred_np, data_range=1.0, channel_axis=-1)
                n += 1

    return {"MSE": mse_t / n, "PSNR": psnr_t / n, "SSIM": ssim_t / n}

In [8]:
def plot_color(
    models_dict, dataset, n=5, device=DEVICE, fname="task3_colorization.png"
):
    indices = np.random.choice(len(dataset), n, replace=False)
    n_cols = 2 + len(models_dict)  # gray + models + gt
    fig, axes = plt.subplots(n, n_cols, figsize=(4 * n_cols, 4 * n))

    titles = ["Grayscale"] + list(models_dict.keys()) + ["Ground Truth"]
    for ax, t in zip(axes[0], titles):
        ax.set_title(t, fontsize=12)

    for row, idx in enumerate(indices):
        sample = dataset[idx]
        gray = torch.tensor(sample["gray_image"])  # (1, H, W)
        gt = sample["image"].transpose(1, 2, 0)  # HWC

        inp = gray.unsqueeze(0).to(device)
        axes[row, 0].imshow(gray.squeeze().numpy(), cmap="gray")

        col = 1
        for model in models_dict.values():
            model.eval()
            with torch.no_grad():
                pred = model(inp).squeeze(0).cpu().clamp(0, 1)
            axes[row, col].imshow(pred.numpy().transpose(1, 2, 0))
            col += 1

        axes[row, col].imshow(gt)

        for ax in axes[row]:
            ax.axis("off")

    plt.tight_layout()
    plt.show()

In [9]:
def task3():
    color_test = ColorizationDataset(train=False)
    loader = torch.utils.data.DataLoader(
        color_test, batch_size=8, shuffle=False, num_workers=2
    )

    unet_c = load("unet_colorization.pkl")

    encdec_path = next(
        (
            p
            for p in ["encdec_colorization.pkl", "encoder_decoder.pkl", "encdec.pkl"]
            if os.path.exists(p)
        ),
        None,
    )
    models = {"U-Net": unet_c}
    if encdec_path:
        models["Enc-Dec"] = load(encdec_path)
    else:
        print("  [!] Encoder-decoder not found, skipping.")

    results = {}
    for name, model in models.items():
        print(f"Evaluating {name}...")
        m = evaluate_color(model, loader)
        results[name] = m
        print(f"  MSE={m['MSE']:.4f}  PSNR={m['PSNR']:.2f} dB  SSIM={m['SSIM']:.4f}")

    plot_color(models, color_test, n=5)
    return results

In [ ]:
t1_acc = task1()
print()
t2_fcn, t2_unet = task2()
print()
t3_results = task3()

print()
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Task 1  ResNet18 test accuracy  : {t1_acc:.2f}%")
print(f"Task 2  FCN-32   mean IoU       : {t2_fcn:.4f}")
print(f"Task 2  U-Net    mean IoU       : {t2_unet:.4f}")
for name, m in t3_results.items():
    print(
        f"Task 3  {name:<8s} PSNR={m['PSNR']:.2f} dB  SSIM={m['SSIM']:.4f}  MSE={m['MSE']:.4f}"
    )
print()
print("Figures saved: task2_masks.png, task3_colorization.png")